In [24]:
import numpy as np
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score,recall_score,precision_score,classification_report
os.chdir("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning")

In [25]:
df=pd.read_csv("train.csv")
df

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699995,699995,29,1,59,6.9,5.2,1.5,26.1,0.88,133,...,Female,Hispanic,Postgraduate,Upper-Middle,Former,Employed,0,0,0,0.0
699996,699996,46,2,72,7.7,7.7,3.8,25.5,0.85,106,...,Female,Hispanic,Graduate,Upper-Middle,Former,Employed,0,0,1,1.0
699997,699997,35,1,50,5.6,6.1,6.4,26.9,0.88,127,...,Female,White,Graduate,Middle,Never,Employed,0,0,0,1.0
699998,699998,49,2,70,5.7,6.9,4.7,25.2,0.86,116,...,Female,White,Highschool,Lower-Middle,Never,Retired,0,0,0,1.0


In [26]:
X,y=df.drop(["diagnosed_diabetes"],axis=1),df["diagnosed_diabetes"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=df['diagnosed_diabetes'])

In [27]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

transf=ColumnTransformer(transformers=[("OHE",ohe, make_column_selector
                                        (dtype_include=object))],remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe=transf.fit_transform(X_train)
X_tst_ohe=transf.transform(X_test)

In [ ]:
solvers=["lbfgs","liblinear","newton-cg","newton-cholesky","sag","saga"]
Cs=np.linspace(0.001,5,20)
scores=[]
for s in solvers:
    for c in Cs:
        lr=LogisticRegression(solver=s,C=c)
        lr.fit(X_trn_ohe,y_train)
        y_pred=lr.predict(X_tst_ohe)
        scores.append([s,c,f1_score(y_test,y_pred,pos_label=1)])
df_scores=pd.DataFrame(scores,columns=["solver","C","score"]) 
df_scores.sort_values("score",ascending=False)

C:\Users\PGCP-AI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\PGCP-AI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://s

In [ ]:
X_ohe=transf.fit_transform(X)
bm=LogisticRegression(solver='lbfgs',C=0.264105)
bm.fit(X_ohe,y)

In [ ]:
test=pd.read_csv("test.csv", index_col=0)
test_ohe=transf.transform(test)
y_pred=bm.predict(test_ohe)
y_pred

In [ ]:
y_pred_prob=bm.predict_proba(test_ohe)


In [ ]:
ss=pd.read_csv("sample_Submission.csv")
ss["diagonsed_diabetes"]=y_pred_prob[:,1]
ss.to_csv("sbt_log_18.4.csv",index=False)
